In [36]:
# =========================
#version00
# =========================

import os
import random
import glob
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F

import matplotlib.pyplot as plt
from korean_lunar_calendar import KoreanLunarCalendar

from copy import deepcopy

from collections import defaultdict


plt.rcParams['font.family'] = 'AppleGothic'  # macOS
plt.rcParams['axes.unicode_minus'] = False

def add_ts_stats(
    df: pd.DataFrame,
    target_col: str = "clipped_SQ",   # 스케일 전(or 스케일 후) 타깃 중 택1
    date_col: str = "영업일자",
    lags = (1, 7, 14, 28),
    roll_windows = (7, 14, 28),
    ewm_spans = (7, 14),
    out_prefix: str = "",
    eps: float = 1e-3
) -> pd.DataFrame:
    """
    시계열 통계 피처 생성 (모두 과거만 사용)
    생성: lag_k, roll_mean_k, roll_std_k, ewm_mean_s, momentum_k, rel_level_k, vol_k
    - momentum_k  = (x_t - x_{t-k}) / (|x_{t-k}|+eps)
    - rel_level_k = x_t / (roll_mean_k + eps)
    - vol_k       = roll_std_k / (roll_mean_k + eps)
    """
    # 정렬 보장
    df = df.sort_values(date_col)
    x = df[target_col].astype("float32")

    # Lags
    for k in lags:
        df[f"{out_prefix}lag_{k}"] = x.shift(k).astype("float32")

    # Rolling mean/std (과거 window, 현재 포함 → 누수 방지 위해 shift(1) 후 rolling도 가능)
    for w in roll_windows:
        base = x.shift(1)  # 현재값 제외
        df[f"{out_prefix}roll_mean_{w}"] = base.rolling(w, min_periods=1).mean().astype("float32")
        df[f"{out_prefix}roll_std_{w}"]  = base.rolling(w, min_periods=1).std().fillna(0).astype("float32")

    # EWMA
    for s in ewm_spans:
        df[f"{out_prefix}ewm_mean_{s}"] = x.ewm(span=s, adjust=False).mean().astype("float32")

    # Momentum & Relative level & Volatility (대표 window=7 사용; 필요시 반복문 확장)
    for k in lags:
        df[f"{out_prefix}momentum_{k}"] = ((x - x.shift(k)) / (np.abs(x.shift(k)) + eps)).astype("float32")
    for w in roll_windows:
        m = df[f"{out_prefix}roll_mean_{w}"]
        s = df[f"{out_prefix}roll_std_{w}"]
        df[f"{out_prefix}rel_level_{w}"] = (x / (m + eps)).astype("float32")      # 수준/평균
        df[f"{out_prefix}vol_{w}"]       = (s / (m + eps)).astype("float32")      # 변동성/평균(무단위)

    return df

#Fixed Random Seed  & Setting Hyperparameter
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


set_seed(42)

LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 16, 50
DEVICE = torch.device("cpu")
MONTH_SCALE = 12

MIN_SEQUENCE_COUNT = 10

# 키: '영업장명_메뉴명', 값: 'YYYY-MM-DD' (ISO 문자열 또는 datetime)
DISCONTINUED = {
    '담하_꼬막_비빔밥': '2024-04-01',
    '담하_들깨_양지탕': '2024-04-01',
    # ...
}

# === 공통 피처 정의 (학습/추론 동일) ===
FEATURES = [
    'clipped_SQ','rolling_mean_7','delta_scaled',
    # 월/연 Fourier (k=1..3)
    'month_sin1','month_cos1','month_sin2','month_cos2','month_sin3','month_cos3',
    'doy_sin1','doy_cos1','doy_sin2','doy_cos2','doy_sin3','doy_cos3',
    'holiday_prox','is_holiday',
    'w_sin1','w_sin2','w_cos1','w_cos2',
    'lag_7','lag_14','lag_28',
    'rel_level_7','rel_level_14',
    'vol_7','vol_14',
    'momentum_7','momentum_14',
    'ewm_mean_7','ewm_mean_14',
    'holiday_prox_lag1', 'holiday_prox_lag2', 'holiday_prox_lag3',
    'holiday_prox_lead1', 'holiday_prox_lead2', 'holiday_prox_lead3'
]

def build_features(
    df: pd.DataFrame,
    scaler_xy: MinMaxScaler,
    scaler_delta: MinMaxScaler,
    fit: bool = False,
    date_col: str = '영업일자'
) -> pd.DataFrame:
    """
    학습/추론에서 동일하게 쓰는 피처 빌더.
    - clipped_SQ, rolling_mean_7 scaling
    - delta_scaled scaling
    - Fourier/holiday proximity/weekly sincos
    - ts-stats (lag/roll/ewm/momentum/rel_level/vol) + 결측 처리
    """
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])

    # 요일/월/시즌
    out['weekday'] = out[date_col].dt.dayofweek.astype(int)
    m = out[date_col].dt.month.astype(np.int16)
    out['season'] = m.map({12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3}).astype(int)

    # 휴일, Fourier, 주간 주기
    out = generate_combined_holiday_list(out, solar_md_holidays, lunar_solar_dates)
    out = add_fourier_seasonal_features(out, date_col=date_col)

    t = (out[date_col] - out[date_col].min()).dt.days.values
    for k in (1, 2):
        out[f'w_sin{k}'] = np.sin(2*np.pi*k*t/7).astype('float32')
        out[f'w_cos{k}'] = np.cos(2*np.pi*k*t/7).astype('float32')

    # holiday proximity (+ lag/lead)
    out = add_holiday_proximity(out, date_col, 'is_holiday', 'holiday_prox', K=7, return_what='prox')
    for k in (1,2,3):
        out[f'holiday_prox_lag{k}']  = out['holiday_prox'].shift(k).fillna(0).astype('float32')
        out[f'holiday_prox_lead{k}'] = out['holiday_prox'].shift(-k).fillna(0).astype('float32')

    # IQR clip / delta / rolling
    out['clipped_SQ']     = clip_iqr(out['매출수량']) if '매출수량' in out.columns else out['clipped_SQ']
    out['delta']          = out['clipped_SQ'].diff().fillna(0)
    out['rolling_mean_7'] = out['clipped_SQ'].rolling(window=7, min_periods=1).mean()

    # scaling
    if fit:
        scaler_xy.fit(out[['clipped_SQ', 'rolling_mean_7']])
        scaler_delta.fit(out[['delta']])
    out[['clipped_SQ', 'rolling_mean_7']] = scaler_xy.transform(out[['clipped_SQ','rolling_mean_7']])
    out[['delta_scaled']] = scaler_delta.transform(out[['delta']])

    # ts-stats (누수 안전 옵션: 현재값 제외하려면 shift(1) 사용)
    # 엄격 모드 예시:
    # out_ts = add_ts_stats(out.copy(), target_col="clipped_SQ", date_col=date_col, ...)
    # 부분만 교체하려면 add_ts_stats 내부에서 rolling을 x.shift(1).rolling(...)로 바꾸세요.
    out = add_ts_stats(out, target_col="clipped_SQ", date_col=date_col,
                       lags=(1,7,14,28), roll_windows=(7,14,28), ewm_spans=(7,14))

    # ✅ 모든 시계열 통계 파생 컬럼에서 NaN/Inf 제거
    ts_cols = [c for c in out.columns if c.startswith((
        'lag_', 'momentum_', 'roll_mean_', 'roll_std_', 'rel_level_', 'vol_', 'ewm_mean_'
    ))]
    out[ts_cols] = (out[ts_cols]
                    .replace([np.inf, -np.inf], np.nan)
                    .fillna(0.0)
                    .astype('float32'))

    return out


def get_lunar_to_solar(years, lunar_month, lunar_day, span=1):
    calendar = KoreanLunarCalendar()
    dates = []
    for year in years:
        for offset in range(-span, span+1):
            try:
                calendar.setLunar(year, lunar_month, lunar_day + offset, False)
                dates.append(calendar.SolarIsoFormat())
            except:
                pass  # 예외 처리: 음력 마지막날 초과
    return dates
# 예시: 2023 ~ 2025
years = [2023, 2024, 2025]
lunar_solar_dates = []
lunar_solar_dates += get_lunar_to_solar(years, 1, 1, span=1)   # 설날 ±1
lunar_solar_dates += get_lunar_to_solar(years, 8, 15, span=1)  # 추석 ±1

solar_md_holidays = [
    (1, 1),   # 신정
    (3, 1),   # 삼일절
    (5, 5),   # 어린이날
    (6, 6),   # 현충일
    (8, 15),  # 광복절
    (10, 3),  # 개천절
    (10, 9),  # 한글날
    (12, 25), # 크리스마스
]

def generate_combined_holiday_list(df, solar_md_list, lunar_solar_list):
    df = df.copy()
    df['영업일자'] = pd.to_datetime(df['영업일자'])

    # 양력 기반 holiday 판별
    df['is_solar_holiday'] = df['영업일자'].apply(
        lambda x: (x.month, x.day) in solar_md_list
    )

    # 음력 변환된 holiday 포함
    lunar_set = set(pd.to_datetime(lunar_solar_list))
    df['is_lunar_holiday'] = df['영업일자'].isin(lunar_set)

    # 최종 통합
    df['is_holiday'] = (df['is_solar_holiday'] | df['is_lunar_holiday']).astype(int)
    df = df.drop(columns=['is_solar_holiday', 'is_lunar_holiday'])
    return df



def remove_leading_zeros_before_sales(df, min_zero_days=90):
    """
    매출 시작 전 연속 0이 일정 기간 이상이면, 그 전 구간 제거
    (단일 메뉴-업장 그룹 DataFrame을 가정)
    """
    sales_started = df['매출수량'] > 0
    if not sales_started.any():
        return df  # 매출이 전혀 없는 경우 그대로 반환

    first_sale_idx = sales_started.idxmax()

    # 매출 시작 전 구간이 충분히 긴 0으로 구성되어 있다면 제거
    df_before = df.loc[:first_sale_idx - 1]
    if len(df_before) >= min_zero_days and (df_before['매출수량'] == 0).all():
        return df.loc[first_sale_idx:]  # 매출 시작부터 반환
    else:
        return df  # 그대로 반환


def _extract_store_name(g: pd.DataFrame) -> str:
    """
    그룹 g에서 업장명 추출:
    - '영업장명' 컬럼이 있으면 그 값을 사용
    - 없으면 '영업장명_메뉴명'에서 첫 '_' 앞을 업장명으로 간주
    """
    if '영업장명' in g.columns:
        return str(g['영업장명'].iloc[0])
    # '영업장명_메뉴명'이 "업장명_메뉴명" 형태라고 가정
    full = str(g['영업장명_메뉴명'].iloc[0])
    return full.split('_', 1)[0]  # '_'가 여러 개여도 첫 구분만 사용


def filter_all_menus_by_leading_zeros(
    train_df: pd.DataFrame,
    min_zero_days: int = 90,
    apply_to_stores: list[str] | None = None,
    exclude_stores: list[str] | None = None,
    group_col: str = '영업장명_메뉴명',
) -> pd.DataFrame:
    """
    모든 메뉴-업장 그룹에 대해 remove_leading_zeros_before_sales를 적용하되,
    특정 업장에만(또는 특정 업장은 제외하고) 적용할 수 있도록 확장.

    Parameters
    ----------
    train_df : 전체 데이터프레임
    min_zero_days : 매출 시작 전 연속 0 최소 일수
    apply_to_stores : 적용 대상 업장명 리스트 (None이면 전 업장 대상)
    exclude_stores : 적용 제외 업장명 리스트 (None이면 제외 없음)
    group_col : 그룹화 기준 컬럼명 (기본: '영업장명_메뉴명')
    """
    parts = []
    apply_set   = set(apply_to_stores) if apply_to_stores is not None else None
    exclude_set = set(exclude_stores)  if exclude_stores  is not None else set()

    # 기존 순서 보존 원하면 sort=False 유지
    for _, g in train_df.groupby(group_col, sort=False):
        store = _extract_store_name(g)

        # 적용 여부 결정
        apply_flag = True
        if apply_set is not None:
            apply_flag = (store in apply_set)
        if store in exclude_set:
            apply_flag = False

        if apply_flag:
            parts.append(remove_leading_zeros_before_sales(g, min_zero_days))
        else:
            parts.append(g)

    if parts:
        return pd.concat(parts, ignore_index=True)
    return train_df.reset_index(drop=True)

# === 추가: Fourier 계절 피처 함수 ===
def add_fourier_seasonal_features(df: pd.DataFrame, date_col: str = '영업일자') -> pd.DataFrame:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    m = df[date_col].dt.month.astype(np.int16)          # 1..12
    doy = df[date_col].dt.dayofyear.astype(np.int16)    # 1..365 (윤년은 무시해도 충분)
    # (A) 월 주기: 12개월 주기, k=1..3 고차 조화항
    for k in (1, 2, 3):
        df[f'month_sin{k}'] = np.sin(2*np.pi*k*m/12).astype('float32')
        df[f'month_cos{k}'] = np.cos(2*np.pi*k*m/12).astype('float32')
    # (B) 연간 주기: 365일 주기, k=1..3 고차 조화항
    for k in (1, 2, 3):
        df[f'doy_sin{k}'] = np.sin(2*np.pi*k*doy/365).astype('float32')
        df[f'doy_cos{k}'] = np.cos(2*np.pi*k*doy/365).astype('float32')
    # (C) 범주형 월 인덱스(임베딩용)
    df['month_idx'] = (m - 1).astype(int)  # 0~11
    return df

def add_holiday_proximity(
    df: pd.DataFrame,
    date_col: str = '영업일자',
    holiday_col: str = 'is_holiday',
    out_col: str = 'holiday_prox',
    K: int = 7,
    return_what: str = 'prox',  # 'prox' 또는 'dist'
) -> pd.DataFrame:
    """
    캘린더 휴일 기준으로 각 날짜가 휴일에 얼마나 근접했는지 계산합니다.
    - prox: (K - min(dist_prev, dist_next)) / K ∈ [0,1], 당일 휴일=1, K일 이상 떨어지면 0
    - dist: min(dist_prev, dist_next) ∈ [0, K] (K로 클리핑)

    Notes
    -----
    * holiday_col은 미래를 '알 수 있는' 캘린더 정보이므로 누수 위험이 없습니다.
    * df의 원래 행 순서를 유지합니다.
    """
    if date_col not in df.columns:
        raise KeyError(f"'{date_col}' not in df")
    if holiday_col not in df.columns:
        raise KeyError(f"'{holiday_col}' not in df")

    # 원래 인덱스 저장
    orig_index = df.index

    # 날짜 정렬본으로 계산
    tmp = df[[date_col, holiday_col]].copy()
    tmp[date_col] = pd.to_datetime(tmp[date_col])
    tmp = tmp.sort_values(date_col)

    mask = tmp[holiday_col].astype(bool)
    # 휴일이면 그 날짜, 아니면 NaT
    s_h = tmp[date_col].where(mask)

    # 과거/미래 휴일 날짜
    prev_h = s_h.ffill()
    next_h = s_h.bfill()

    # 거리 계산(일수)
    dist_prev = (tmp[date_col] - prev_h).dt.days.astype('float32')
    dist_next = (next_h - tmp[date_col]).dt.days.astype('float32')

    # 휴일이 아예 없을 때 NaN → K+1로 대체
    dist_prev = dist_prev.fillna(K + 1)
    dist_next = dist_next.fillna(K + 1)

    # 최소 거리 후 K로 클리핑
    dist_h = np.minimum(dist_prev, dist_next).clip(0, K).astype('float32')

    if return_what == 'dist':
        out = dist_h
    elif return_what == 'prox':
        # 근접도: 0(멀다) ~ 1(당일 휴일)
        out = ((K - dist_h) / K).astype('float32')
    else:
        raise ValueError("return_what must be 'prox' or 'dist'")

    # 정렬 전 순서로 복원
    out = out.reindex(tmp.index)                # 안전: 이미 tmp와 동일
    out_df = pd.DataFrame({out_col: out}, index=tmp.index)
    out_df = out_df.reindex(orig_index)         # 원래 df 순서로

    # 원본 df에 컬럼으로 추가
    df[out_col] = out_df[out_col].values.astype('float32')
    return df

def ensure_time_major(x: torch.Tensor, lookback: int, n_features: int) -> torch.Tensor:
    """
    x shape을 (B, T, F)로 강제 정렬.
    - 올바르면 그대로 반환
    - (B, F, T)로 뒤집혀 있으면 permute(0,2,1)
    """
    if x.dim() != 3:
        raise ValueError(f"Expected 3D tensor, got {x.dim()}D: {tuple(x.shape)}")
    B, A, B2 = x.shape
    # 정상: (B, T, F)
    if A == lookback and B2 == n_features:
        return x
    # 뒤집힘: (B, F, T)
    if A == n_features and B2 == lookback:
        return x.permute(0, 2, 1).contiguous()
    # 그 외: 명시 에러로 빨리 잡기
    raise ValueError(f"Unexpected shape for sequence tensor: {tuple(x.shape)} (T={lookback}, F={n_features})")

#Data load
train = pd.read_csv('./train/train.csv')
train = generate_combined_holiday_list(train, solar_md_holidays, lunar_solar_dates)
train = filter_all_menus_by_leading_zeros(
    train,
    min_zero_days=90,
    apply_to_stores=['담하','라그로타','미라시아' ]  # 여기에 대상 업장명만 나열
)

#Define Model
class MultiEmbeddingLSTM(nn.Module):
    def __init__(
        self,
        input_dim,                  # 수치 feature 수
        hidden_dim=64,
        num_layers=2,
        output_dim=7,
        num_weekdays=7,
        weekday_embed_dim=3,
        num_seasons=4,
        season_embed_dim=2,
        dropout=0.3
    ):
        super().__init__()

        # 임베딩
        self.weekday_embedding = nn.Embedding(num_weekdays, weekday_embed_dim)
        self.season_embedding = nn.Embedding(num_seasons, season_embed_dim)

        # LayerNorm for embedding
        self.weekday_norm = nn.LayerNorm(weekday_embed_dim)
        self.season_norm = nn.LayerNorm(season_embed_dim)

        # Dropout after embedding
        self.embedding_dropout = nn.Dropout(dropout)

        total_input_dim = input_dim + weekday_embed_dim + season_embed_dim

        # LSTM
        self.lstm = nn.LSTM(total_input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)

        # LSTM output dropout
        self.post_lstm_dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, weekday_ids, season_ids):
        """
        x: (B, T, input_dim)
        weekday_ids: (B, T)
        season_ids: (B, T)
        """
        weekday_embed = self.weekday_embedding(weekday_ids)      # (B, T, D2)
        season_embed = self.season_embedding(season_ids)         # (B, T, D3)

        # 정규화
        weekday_embed = self.weekday_norm(weekday_embed)
        season_embed = self.season_norm(season_embed)

        x_concat = torch.cat([x, weekday_embed, season_embed], dim=-1)  # (B, T, total_dim)
        x_concat = self.embedding_dropout(x_concat)

        out, _ = self.lstm(x_concat)
        out = self.post_lstm_dropout(out)

        return self.fc(out[:, -1, :])
    
def clip_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    return np.clip(series, None, upper)

def compute_iqr_lower_bounds(train_df):
    lower_bounds = {}
    for menu, group in train_df.groupby('영업장명_메뉴명'):
        q1 = group['매출수량'].quantile(0.25)
        q3 = group['매출수량'].quantile(0.75)
        iqr = q3 - q1
        lower = max(q1 - 1.5 * iqr, 0)
        menu_key = menu[0] if isinstance(menu, tuple) else menu
        lower_bounds[menu_key] = lower
    return lower_bounds

from numpy.lib.stride_tricks import sliding_window_view

def train_lstm(train_df, use_validation=True, dropout=0.3):
    trained_models = {}

    for store_menu, group in tqdm(train_df.groupby(['영업장명_메뉴명'], sort=False), desc='Training LSTM'):
        # key를 문자열로 고정(파일명/딕셔너리 키 혼선 방지)
        key = store_menu if isinstance(store_menu, str) else "_".join(map(str, store_menu))

        store_train = group.sort_values('영업일자').copy()
        store_train['영업일자'] = pd.to_datetime(store_train['영업일자'])

        # 짧은 시계열 제외
        if len(store_train) < LOOKBACK + PREDICT + MIN_SEQUENCE_COUNT:
            continue

        # ---- 스플릿: 행 기준으로 먼저 나누고 train-part로만 scaler fit ----
        N = len(store_train)
        if use_validation:
            cutoff_row = max(LOOKBACK, int(round(N * 0.8)))
            cutoff_row = min(cutoff_row, N-1)
        else:
            cutoff_row = N

        scaler_xy    = MinMaxScaler()   # ['clipped_SQ','rolling_mean_7']를 "같이" fit/transform
        scaler_delta = MinMaxScaler()   # ['delta'] 전용

        # 1) train-part로 먼저 fit
        train_part = store_train.iloc[:cutoff_row].copy()
        _ = build_features(train_part, scaler_xy, scaler_delta, fit=True, date_col='영업일자')

        # 2) 같은 스케일러로 전체 transform (누수 없음)
        ft = build_features(store_train, scaler_xy, scaler_delta, fit=False, date_col='영업일자')

        # ---- 시퀀스화: 벡터화 (경고/속도 개선) ----
        vals = ft[FEATURES].values.astype(np.float32)   # (N, F)
        tgt  = ft['clipped_SQ'].values.astype(np.float32)
        wd   = ft['weekday'].values.astype(np.int64)
        ss   = ft['season'].values.astype(np.int64)

        total_seq = len(ft) - LOOKBACK - PREDICT + 1
        if total_seq <= 0:
            continue

        # X: (S, LOOKBACK, F), y: (S, PREDICT), wd/ss: (S, LOOKBACK)
        X_np = sliding_window_view(vals, LOOKBACK, axis=0)[:total_seq]
        y_np = sliding_window_view(tgt, LOOKBACK + PREDICT, axis=0)[:total_seq, LOOKBACK:]
        wd_np = sliding_window_view(wd, LOOKBACK, axis=0)[:total_seq]
        ss_np = sliding_window_view(ss, LOOKBACK, axis=0)[:total_seq]

        X = torch.from_numpy(X_np).float()
        X = ensure_time_major(X, LOOKBACK, len(FEATURES))
        y = torch.from_numpy(y_np).float()
        weekday_seqs = torch.from_numpy(wd_np).long()
        season_seqs  = torch.from_numpy(ss_np).long()

        # ---- 시퀀스 기준 split ----
        if use_validation:
            split_idx = int(len(X) * 0.8)
            X_train, X_val = X[:split_idx], X[split_idx:]
            y_train, y_val = y[:split_idx], y[split_idx:]
            weekday_train, weekday_val = weekday_seqs[:split_idx], weekday_seqs[split_idx:]
            season_train,  season_val  = season_seqs[:split_idx],  season_seqs[split_idx:]
        else:
            X_train, y_train = X, y
            weekday_train, season_train = weekday_seqs, season_seqs
            X_val = y_val = weekday_val = season_val = None

        X_train, y_train = X_train.to(DEVICE), y_train.to(DEVICE)
        weekday_train, season_train = weekday_train.to(DEVICE), season_train.to(DEVICE)
        if use_validation and X_val is not None:
            X_val, y_val = X_val.to(DEVICE), y_val.to(DEVICE)
            weekday_val, season_val = weekday_val.to(DEVICE), season_val.to(DEVICE)

        # ---- 모델 ----
        model = MultiEmbeddingLSTM(input_dim=len(FEATURES), output_dim=PREDICT, dropout=dropout).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()

        train_losses, val_losses = [], []
        for epoch in range(EPOCHS):
            model.train()
            total_loss, n_batches = 0.0, 0
            idx = torch.randperm(len(X_train))

            for i in range(0, len(X_train), BATCH_SIZE):
                b = idx[i:i+BATCH_SIZE]
                out = model(X_train[b], weekday_train[b], season_train[b])
                loss = criterion(out, y_train[b])
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                total_loss += loss.item(); n_batches += 1

            train_losses.append(total_loss / max(1, n_batches))

            if use_validation and X_val is not None and len(X_val) > 0:
                model.eval()
                with torch.no_grad():
                    val_out = model(X_val, weekday_val, season_val)
                    val_losses.append(criterion(val_out, y_val).item())
        visualize_loss(train_losses, val_losses if use_validation else None,
               key, save=True, show=False, verbose=True)

        # 동일 key로 저장
        trained_models[key] = {
            'model': model.eval(),
            'scaler_xy': scaler_xy,
            'scaler_delta': scaler_delta,
            'last_sequence': {
                'X': ft[FEATURES].values[-LOOKBACK:],
                'weekday': ft['weekday'].values[-LOOKBACK:],
                'season':  ft['season'].values[-LOOKBACK:]
            }
        }

    return trained_models


def visualize_loss(
    train_losses,
    val_losses,
    store_menu,
    save=False,
    out_dir="./loss_plots",
    show=False,               # 노트북에 바로 표시하려면 True
    verbose=False             # 길이/유효값 로그 출력
):
    import numpy as np
    plt.figure(figsize=(6,4))

    # 리스트/텐서 → float 배열 변환
    def to_float_array(xs):
        if xs is None: 
            return np.array([], dtype=float)
        # torch 텐서나 리스트 섞여 있어도 안전 변환
        try:
            arr = np.asarray([float(x) for x in xs], dtype=float)
        except Exception:
            arr = np.array(xs, dtype=float)
        return arr

    tr = to_float_array(train_losses)
    va = to_float_array(val_losses)

    # 유한값만 마스크
    tr_mask = np.isfinite(tr)
    va_mask = np.isfinite(va)

    drew_any = False

    if tr.size > 0 and tr_mask.any():
        x_tr = np.arange(1, tr.size + 1)[tr_mask]
        y_tr = tr[tr_mask]
        plt.plot(x_tr, y_tr, marker='o', linewidth=1.5, label='Train Loss')
        drew_any = True

    if va.size > 0 and va_mask.any():
        x_va = np.arange(1, va.size + 1)[va_mask]
        y_va = va[va_mask]
        plt.plot(x_va, y_va, marker='o', linewidth=1.5, label='Validation Loss')
        drew_any = True

    title = f"[{store_menu}] Train vs Validation Loss"
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    if drew_any:
        plt.legend()
        plt.grid(True, alpha=0.4)
        # 축 범위가 너무 타이트하면 살짝 여유
        ymin = min(np.min(tr[tr_mask]) if tr_mask.any() else np.inf,
                   np.min(va[va_mask]) if va_mask.any() else np.inf)
        ymax = max(np.max(tr[tr_mask]) if tr_mask.any() else -np.inf,
                   np.max(va[va_mask]) if va_mask.any() else -np.inf)
        if np.isfinite(ymin) and np.isfinite(ymax) and ymin != ymax:
            pad = 0.05 * (ymax - ymin)
            plt.ylim(ymin - pad, ymax + pad)
    else:
        # 아무 것도 못 그릴 때, 이유를 그림에 표기
        msg = "No points to plot"
        if tr.size == 0 and (va is None or va.size == 0):
            msg += " (empty train/val lists)"
        elif (tr.size > 0 and not tr_mask.any()) and (va.size == 0 or not va_mask.any()):
            msg += " (all values are NaN/Inf)"
        plt.grid(True, alpha=0.4)
        plt.text(0.5, 0.5, msg, ha='center', va='center', transform=plt.gca().transAxes, fontsize=12)

    # 파일명 안전 처리
    name_str = store_menu if isinstance(store_menu, str) else "_".join(map(str, store_menu))
    safe_name = re.sub(r'[^\w\-_.]', '_', name_str)

    if verbose:
        print(f"[visualize_loss] {safe_name}: "
              f"len(train)={len(tr)}, finite(train)={tr_mask.sum()}, "
              f"len(val)={len(va)}, finite(val)={va_mask.sum()}, drew={drew_any}")

    if save:
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, f"{safe_name}.png")
        plt.tight_layout()
        plt.savefig(path, dpi=150, bbox_inches='tight')
    if show and not save:
        plt.tight_layout()
        plt.show()

    plt.close()


def inverse_clipped_from_scaler(scaler_xy: MinMaxScaler, scaled_vals: np.ndarray) -> np.ndarray:
    """
    scaler_xy는 ['clipped_SQ','rolling_mean_7']에 대해 fit 되어 있음.
    clipped_SQ만 역변환하려면 2열 dummy를 만들어 1열만 복원.
    """
    dummy = np.zeros((len(scaled_vals), 2), dtype=np.float32)
    dummy[:, 0] = scaled_vals
    inv = scaler_xy.inverse_transform(dummy)[:, 0]
    return inv

#Prediction
def predict_lstm(test_df, trained_models, test_prefix: str, lower_bound_dict: dict):
    results = []

    for store_menu, store_test in test_df.groupby(['영업장명_메뉴명'], sort=False):
        key = store_menu if isinstance(store_menu, str) else "_".join(map(str, store_menu))
        if key not in trained_models:
            continue

        model        = trained_models[key]['model']
        scaler_xy    = trained_models[key]['scaler_xy']
        scaler_delta = trained_models[key]['scaler_delta']

        store_test_sorted = store_test.sort_values('영업일자').copy()
        store_test_sorted['영업일자'] = pd.to_datetime(store_test_sorted['영업일자'])

        # 학습과 동일 전처리 (fit=False)
        ft = build_features(store_test_sorted, scaler_xy, scaler_delta, fit=False, date_col='영업일자')

        # 최근 LOOKBACK 시퀀스 확보
        if len(ft) < LOOKBACK:
            last_seq   = trained_models[key]['last_sequence']
            x_input    = torch.tensor([last_seq['X']]).float().to(DEVICE)
            x_input = ensure_time_major(x_input, LOOKBACK, len(FEATURES))  # ★
            weekday_seq= torch.tensor([last_seq['weekday']]).long().to(DEVICE)
            season_seq = torch.tensor([last_seq['season']]).long().to(DEVICE)
        else:
            recent     = ft.iloc[-LOOKBACK:]
            x_input    = torch.tensor([recent[FEATURES].values]).float().to(DEVICE)
            x_input = ensure_time_major(x_input, LOOKBACK, len(FEATURES))  # ★
            weekday_seq= torch.tensor([recent['weekday'].values]).long().to(DEVICE)
            season_seq = torch.tensor([recent['season'].values]).long().to(DEVICE)

        # 예측 (스케일 공간)
        with torch.no_grad():
            pred_scaled = model(x_input, weekday_seq, season_seq).squeeze().cpu().numpy()

        # 역정규화 + 하한 클리핑(메뉴별 하한 없으면 1)
        restored = inverse_clipped_from_scaler(scaler_xy, pred_scaled)
        lower_bound = lower_bound_dict.get(key, 1)
        restored = np.maximum(restored, 1.0)

        # 제출 포맷
        pred_dates = [f"{test_prefix}+{i+1}일" for i in range(PREDICT)]
        for d, val in zip(pred_dates, restored):
            results.append({
                '영업일자': d,
                '영업장명_메뉴명': key,      # ★ 전체 문자열로 저장
                '매출수량': float(val)
            })

    return pd.DataFrame(results)

def convert_to_submission_format(pred_df: pd.DataFrame, sample_submission: pd.DataFrame):
    # (영업일자, 메뉴) → 매출수량 딕셔너리로 변환
    pred_dict = dict(zip(
        zip(pred_df['영업일자'], pred_df['영업장명_메뉴명']),
        pred_df['매출수량']
    ))

    final_df = sample_submission.copy()

    for row_idx in final_df.index:
        date = final_df.loc[row_idx, '영업일자']
        for col in final_df.columns[1:]:  # 메뉴명들
            final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)

    return final_df


In [33]:
trained_models = train_lstm(train, use_validation=True, dropout=0.1)

Training LSTM:   1%|          | 1/193 [00:09<29:10,  9.12s/it]

[visualize_loss] 느티나무_셀프BBQ_1인_수저세트: len(train)=50, finite(train)=50, len(val)=50, finite(val)=50, drew=True


Training LSTM:   1%|          | 2/193 [00:18<29:04,  9.13s/it]

[visualize_loss] 느티나무_셀프BBQ_BBQ55_단체_: len(train)=50, finite(train)=50, len(val)=50, finite(val)=50, drew=True


Training LSTM:   2%|▏         | 3/193 [00:27<28:38,  9.04s/it]

...

[visualize_loss] 화담숲카페_아메리카노_ICE: len(train)=50, finite(train)=50, len(val)=50, finite(val)=50, drew=True


Training LSTM:  99%|█████████▉| 192/193 [26:11<00:08,  8.92s/it]

[visualize_loss] 화담숲카페_카페라떼_ICE: len(train)=50, finite(train)=50, len(val)=50, finite(val)=50, drew=True


Training LSTM: 100%|██████████| 193/193 [26:20<00:00,  8.19s/it]

[visualize_loss] 화담숲카페_현미뻥스크림: len(train)=50, finite(train)=50, len(val)=50, finite(val)=50, drew=True


In [37]:
all_preds = []

# 모든 test_*.csv 순회
test_files = sorted(glob.glob('./test/TEST_*.csv'))
df = pd.read_csv('./train/train.csv')
lower_bound_dict = compute_iqr_lower_bounds(df)
for path in test_files:
    test_df = pd.read_csv(path)
    # 파일명에서 접두어 추출 (예: TEST_00)
    filename = os.path.basename(path)
    test_prefix = re.search(r'(TEST_\d+)', filename).group(1)

    pred_df = predict_lstm(test_df, trained_models, test_prefix, lower_bound_dict)
    all_preds.append(pred_df)
    
full_pred_df = pd.concat(all_preds, ignore_index=True)

In [38]:
sample_submission = pd.read_csv('./sample_submission.csv')
submission = convert_to_submission_format(full_pred_df, sample_submission)
submission.to_csv('./Prediction/model_v7_0.csv', index=False, encoding='utf-8-sig')
result = pd.read_csv('./Prediction/model_v7_0.csv')
display(result.head())

/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_4831/336480522.py:778: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.2308883666992188' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_4831/336480522.py:778: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '8.318193435668945' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_4831/336480522.py:778: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4.61230993270874' has dtype inco

,영업일자,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ_BBQ55(단체),"느티나무 셀프BBQ_대여료 30,000원","느티나무 셀프BBQ_대여료 60,000원","느티나무 셀프BBQ_대여료 90,000원","느티나무 셀프BBQ_본삼겹 (단품,실내)",느티나무 셀프BBQ_스프라이트 (단체),느티나무 셀프BBQ_신라면,느티나무 셀프BBQ_쌈야채세트,...,화담숲주막_스프라이트,화담숲주막_참살이 막걸리,화담숲주막_찹쌀식혜,화담숲주막_콜라,화담숲주막_해물파전,화담숲카페_메밀미숫가루,화담숲카페_아메리카노 HOT,화담숲카페_아메리카노 ICE,화담숲카페_카페라떼 ICE,화담숲카페_현미뻥스크림
0,TEST_00+1일,3.230888,1.000000,8.318193,4.61231,1.0,1.0,4.969944,1.000000,1.427957,...,4.954815,9.474245,14.798829,1.905724,36.972668,9.484403,3.073032,15.938519,3.234250,5.664501
1,TEST_00+2일,6.838723,1.002685,3.459318,1.00000,1.0,1.0,17.446274,1.399614,1.000000,...,1.000000,2.845379,2.711002,1.000000,1.311781,1.000000,1.582667,1.000000,1.000000,1.000000
2,TEST_00+3일,4.587320,25.598133,1.471952,1.00000,1.0,1.0,15.181211,2.524719,1.000000,...,1.540719,5.313407,5.332724,3.540042,22.367752,13.279286,3.574736,25.329031,2.517555,4.395793
3,TEST_00+4일,6.871662,1.000000,1.000000,1.00000,1.0,1.0,7.219997,1.939924,1.000000,...,2.269628,4.182890,6.789391,3.058056,17.236771,18.252745,3.693020,16.719507,2.217984,2.533826
4,TEST_00+5일,8.481396,50.310940,1.545647,1.00000,1.0,1.0,4.505751,1.000000,1.000000,...,1.000000,1.973272,7.353449,1.093391,6.445723,7.774331,2.135460,22.239046,3.249308,1.494368
